<a href="https://colab.research.google.com/github/Lkarthikeya/Fake-News-Detection/blob/main/Another_copy_of_Fake_News_Detection_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =====================================================
# 📘 Fake News Detection using NLP (Final Project)
# Author: Karthikeya (CSE - AIML, 2023–2027)
# Internship: YBI Foundation - AI & Generative AI
# =====================================================

# 🪜 Step 1 — Import Libraries
import pandas as pd
import numpy as np
import string
import nltk
from nltk.corpus import stopwords
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import pickle
import zipfile
import os

# 🪜 Step 2 — Download NLTK Data
nltk.download('stopwords')

from google.colab import files
uploaded = files.upload()


# 🪜 Step 3 — Extract the uploaded archive.zip
print("📦 Extracting your dataset...")
zip_name = "archive.zip"
if not os.path.exists(zip_name):
    zip_name = "archive (1).zip"

with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall("dataset")

print("✅ Extracted files:", os.listdir("dataset"))

# 🪜 Step 4 — Load and Prepare Dataset
fake = pd.read_csv("/content/dataset/Fake.csv")
true = pd.read_csv("/content/dataset/True.csv")

fake["label"] = 0
true["label"] = 1

df = pd.concat([fake, true], axis=0)
df = df.sample(frac=1).reset_index(drop=True)

print("✅ Dataset loaded successfully!")
print(df.head())

# 🪜 Step 5 — Clean the Text (Fast Optimized Version)
stop_words = set(stopwords.words("english"))

def clean_text(text):
    text = text.lower()
    text = text.translate(str.maketrans("", "", string.punctuation))
    words = [word for word in text.split() if word not in stop_words]
    return " ".join(words)

df["clean_text"] = df["text"].apply(clean_text)
print("✅ Text cleaning done!")

# 🪜 Step 6 — TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
x = tfidf.fit_transform(df["clean_text"])
y = df["label"]

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("✅ TF-IDF Vectorization complete!")
print("Training data shape:", x_train.shape)
print("Testing data shape:", x_test.shape)

# 🪜 Step 7 — Train the Logistic Regression Model
model = LogisticRegression(max_iter=1000)
model.fit(x_train, y_train)
print("✅ Model training complete!")

# 🪜 Step 8 — Evaluate the Model
y_pred = model.predict(x_test)
print("\n🎯 Accuracy:", accuracy_score(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))

# 🪜 Step 9 — Mount Google Drive to Save Model
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# 🪜 Step 10 — Save Model and TF-IDF Vectorizer Permanently
with open("fake_news_model.pkl", "wb") as f:
    pickle.dump(model, f)

with open("tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

!cp fake_news_model.pkl /content/drive/MyDrive/
!cp tfidf_vectorizer.pkl /content/drive/MyDrive/
print("✅ Model and TF-IDF vectorizer saved permanently to Google Drive!")

# 🪜 Step 11 — Prediction Function
def predict_news(text):
    x_input = tfidf.transform([text])
    pred = model.predict(x_input)[0]
    prob = model.predict_proba(x_input)[0]
    label = "🚨 Fake News" if pred == 0 else "✅ Real News"
    confidence = max(prob) * 100
    print(f"\n📰 Input: {text}")
    print(f"Prediction: {label}")
    print(f"Confidence: {confidence:.2f}%")

# 🪜 Step 12 — Test Examples
predict_news("Government launches new AI initiative for rural education.")
predict_news("Donald Trump secretly runs alien base in Nevada desert.")

# 🪜 Step 13 — Interactive Input
news_input = input("\n✍️ Enter any news headline or paragraph to test: ")
predict_news(news_input)

print("\n✅ Your Fake News Detection Model is working perfectly! 🚀")


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


Saving archive.zip to archive.zip
📦 Extracting your dataset...
✅ Extracted files: ['Fake.csv', 'True.csv']
✅ Dataset loaded successfully!
                                               title  \
0  Bangladesh says agreed with Myanmar for UNHCR ...   
1   WATCH: Paul Manafort Gets DESTROYED On CNN Fo...   
2  House speaker optimistic on tax reform prospec...   
3  Putin says Democrats wrongly trying to blame o...   
4  FURIOUS FBI Agents Speak Out On Clinton Email ...   

                                                text       subject  \
0  DHAKA (Reuters) - Bangladesh and Myanmar have ...     worldnews   
1  Paul Manafort, the chair of Donald Trump s cam...          News   
2  WASHINGTON (Reuters) - The top Republican in t...  politicsNews   
3  MOSCOW (Reuters) - Russian President Vladimir ...  politicsNews   
4  FBI Agents are coming forward now to voice the...      politics   

                 date  label  
0  November 25, 2017       1  
1     August 14, 2016      0  
2      June

ValueError: mount failed